<a href="https://colab.research.google.com/github/abdullahhadi9898/-indata-software-sentiment/blob/main/indata_sentiment_analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#importing data from google drive to collab

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Cell 4 — Verify files are accessible
import os

# Update this path to match your actual folder name in Google Drive
DRIVE_PATH = '/content/drive/MyDrive/indata-project'

# List files in the folder
files = os.listdir(DRIVE_PATH)
print("Files found in your Google Drive folder:")
for f in files:
    size = os.path.getsize(f'{DRIVE_PATH}/{f}') / (1024**3)
    print(f"  {f}  —  {size:.2f} GB")

Files found in your Google Drive folder:
  Software.jsonl.gz  —  0.46 GB
  meta_Software.jsonl.gz  —  0.06 GB


In [5]:
# Cell 5 — Set file paths
REVIEW_FILE = '/content/drive/MyDrive/indata-project/Software.jsonl.gz'
META_FILE = '/content/drive/MyDrive/indata-project/meta_Software.jsonl.gz'
SAMPLE_SIZE = 20000  # 20k per star rating = 100k total

print("✅ File paths set")
print(f"   Review file : {REVIEW_FILE}")
print(f"   Meta file   : {META_FILE}")

✅ File paths set
   Review file : /content/drive/MyDrive/indata-project/Software.jsonl.gz
   Meta file   : /content/drive/MyDrive/indata-project/meta_Software.jsonl.gz


In [6]:
# Cell 6 — Load raw review data
import json
import gzip

print("Loading reviews... this will take a few minutes.")
print("You will see progress updates every 500,000 rows.\n")

reviews = []
with gzip.open(REVIEW_FILE, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            reviews.append(json.loads(line.strip()))
        except json.JSONDecodeError:
            continue

        if i % 500000 == 0 and i > 0:
            print(f"  Scanned {i:,} rows so far...")

df_raw = pd.DataFrame(reviews)

print(f"\n✅ Raw data loaded successfully")
print(f"   Total rows    : {df_raw.shape[0]:,}")
print(f"   Total columns : {df_raw.shape[1]}")
print(f"\nColumns found:")
print(df_raw.columns.tolist())

Loading reviews... this will take a few minutes.
You will see progress updates every 500,000 rows.

  Scanned 500,000 rows so far...
  Scanned 1,000,000 rows so far...
  Scanned 1,500,000 rows so far...
  Scanned 2,000,000 rows so far...
  Scanned 2,500,000 rows so far...
  Scanned 3,000,000 rows so far...
  Scanned 3,500,000 rows so far...
  Scanned 4,000,000 rows so far...
  Scanned 4,500,000 rows so far...


NameError: name 'pd' is not defined

importing libraries

In [8]:
# ============================================================
# MASTER CELL — Run this once to set up everything
# Install → Import → Load → Sample → Clean
# ============================================================

import subprocess
subprocess.run(['pip', 'install', 'vaderSentiment', 'gensim', 'wordcloud', '-q'])

import json, gzip, re, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from collections import defaultdict
import random

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from gensim import corpora
from gensim.models import LdaModel
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)
plt.style.use('seaborn-v0_8-whitegrid')

# ── File paths ──────────────────────────────────────────────
REVIEW_FILE = '/content/drive/MyDrive/indata-project/Software.jsonl.gz'
META_FILE   = '/content/drive/MyDrive/indata-project/meta_Software.jsonl.gz'
SAMPLE_SIZE = 20000
SAVE_PATH   = '/content/drive/MyDrive/indata-project/software_reviews_final.csv'

print("✅ Libraries ready")
print("=" * 50)

# ── STEP 1: Load and sample simultaneously ───────────────────
print("STEP 1 — Loading and sampling...")
random.seed(42)
buckets = defaultdict(list)

with gzip.open(REVIEW_FILE, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            record = json.loads(line.strip())
        except json.JSONDecodeError:
            continue
        rating = record.get('rating')
        if rating not in [1.0, 2.0, 3.0, 4.0, 5.0]:
            continue
        bucket = buckets[rating]
        if len(bucket) < SAMPLE_SIZE:
            bucket.append(record)
        else:
            j = random.randint(0, i)
            if j < SAMPLE_SIZE:
                bucket[j] = record
        if i % 500000 == 0 and i > 0:
            print(f"  Scanned {i:,} rows...")

all_reviews = []
for rating, records in buckets.items():
    all_reviews.extend(records)

df = pd.DataFrame(all_reviews).reset_index(drop=True)
print(f"✅ Loaded: {len(df):,} rows")

# ── STEP 2: Clean and filter ─────────────────────────────────
print("\nSTEP 2 — Cleaning and filtering...")

COLS = ['rating', 'title', 'text', 'parent_asin',
        'timestamp', 'helpful_vote', 'verified_purchase']
df = df[COLS].copy()

before = len(df)
df = df[df['verified_purchase'] == True].reset_index(drop=True)
print(f"  Verified filter   : {before:,} → {len(df):,}")

before = len(df)
df = df[df['text'].notna() & (df['text'].str.strip() != '')].reset_index(drop=True)
print(f"  Empty text filter : {before:,} → {len(df):,}")

before = len(df)
df = df[df['text'].str.len() >= 20].reset_index(drop=True)
print(f"  Min length filter : {before:,} → {len(df):,}")

df['date']       = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
df['year']       = df['date'].dt.year
df['month']      = df['date'].dt.month
df['year_month'] = df['date'].dt.to_period('M')
df['helpful_vote'] = df['helpful_vote'].fillna(0).astype(int)

print(f"✅ Clean dataset: {len(df):,} rows")

# ── STEP 3: Load and join metadata ───────────────────────────
print("\nSTEP 3 — Loading metadata...")

meta = []
with gzip.open(META_FILE, 'rt', encoding='utf-8') as f:
    for line in f:
        try:
            meta.append(json.loads(line.strip()))
        except json.JSONDecodeError:
            continue

df_meta = pd.DataFrame(meta)
print(f"  Meta loaded: {len(df_meta):,} products")

META_COLS = ['parent_asin', 'title', 'store', 'price',
             'average_rating', 'rating_number']
meta_cols_exist = [c for c in META_COLS if c in df_meta.columns]
df_meta = df_meta[meta_cols_exist].drop_duplicates('parent_asin')

df = df.merge(df_meta, on='parent_asin', how='left')
df = df.rename(columns={'title_x': 'review_title', 'title_y': 'product_name'})

print(f"✅ Merged dataset: {len(df):,} rows, {df.shape[1]} columns")

# ── STEP 4: Save to Google Drive ─────────────────────────────
print("\nSTEP 4 — Saving to Google Drive...")
df.to_csv(SAVE_PATH, index=False)
size = os.path.getsize(SAVE_PATH) / (1024**2)
print(f"✅ Saved: {SAVE_PATH}")
print(f"   File size: {size:.1f} MB")

# ── FINAL SUMMARY ────────────────────────────────────────────
print("\n" + "=" * 50)
print("DATASET READY")
print("=" * 50)
print(f"Total reviews   : {len(df):,}")
print(f"Columns         : {df.shape[1]}")
print(f"Date range      : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique products : {df['parent_asin'].nunique():,}")
if 'store' in df.columns:
    print(f"Unique brands   : {df['store'].nunique():,}")
print(f"\nRating distribution:")
print(df['rating'].value_counts().sort_index())
print(f"\nColumns available:")
print(df.columns.tolist())

✅ Libraries ready
STEP 1 — Loading and sampling...
  Scanned 500,000 rows...
  Scanned 1,000,000 rows...
  Scanned 1,500,000 rows...
  Scanned 2,000,000 rows...
  Scanned 2,500,000 rows...
  Scanned 3,000,000 rows...
  Scanned 3,500,000 rows...
  Scanned 4,000,000 rows...
  Scanned 4,500,000 rows...
✅ Loaded: 100,000 rows

STEP 2 — Cleaning and filtering...
  Verified filter   : 100,000 → 94,135
  Empty text filter : 94,135 → 94,129
  Min length filter : 94,129 → 75,851
✅ Clean dataset: 75,851 rows

STEP 3 — Loading metadata...
  Meta loaded: 89,251 products
✅ Merged dataset: 75,851 rows, 16 columns

STEP 4 — Saving to Google Drive...
✅ Saved: /content/drive/MyDrive/indata-project/software_reviews_final.csv
   File size: 24.5 MB

DATASET READY
Total reviews   : 75,851
Columns         : 16
Date range      : 2000-05-25 → 2023-09-03
Unique products : 15,301
Unique brands   : 7,541

Rating distribution:
rating
1.0    14655
2.0    15574
3.0    14843
4.0    15680
5.0    15099
Name: count, dt

In [9]:
df = pd.read_csv('/content/drive/MyDrive/indata-project/software_reviews_final.csv')

In [10]:
print("Building text cleaning pipeline...")

# Set up tools
stop_words  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

# Add custom stop words specific to software reviews
# These words appear in almost every review and add no topic signal
custom_stops = {
    'software', 'product', 'program', 'app', 'application',
    'use', 'used', 'using', 'work', 'works', 'worked',
    'get', 'got', 'one', 'also', 'would', 'could', 'even',
    'time', 'like', 'just', 'really', 'very', 'good', 'great',
    'bad', 'well', 'make', 'made', 'need', 'want', 'way',
    'much', 'many', 'lot', 'thing', 'things', 'purchase',
    'bought', 'buy', 'amazon', 'review', 'star', 'rating'
}
stop_words.update(custom_stops)

def clean_text(text):
    # Step 1 — lowercase
    text = str(text).lower()

    # Step 2 — remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Step 3 — remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # Step 4 — remove numbers and special characters
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)

    # Step 5 — tokenise (split into individual words)
    tokens = text.split()

    # Step 6 — remove stop words and very short words
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    # Step 7 — lemmatise (convert to base form)
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return ' '.join(tokens)

print("Cleaning 75,851 reviews... this takes 2-3 minutes")

df['clean_text'] = df['text'].apply(clean_text)

# Remove rows where cleaning left nothing
before = len(df)
df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)
print(f"Rows after empty clean_text removal: {len(df):,}")

print(f"\n✅ Text cleaning complete")
print(f"\nOriginal text sample:")
print(df['text'].iloc[0][:200])
print(f"\nCleaned text sample:")
print(df['clean_text'].iloc[0][:200])

Building text cleaning pipeline...
Cleaning 75,851 reviews... this takes 2-3 minutes
Rows after empty clean_text removal: 75,593

✅ Text cleaning complete

Original text sample:
Uninstalled right away, was awkwardly set up

Cleaned text sample:
uninstalled right away awkwardly set


In [11]:
#VADER Sentiment Scoring
# Run on ORIGINAL text — not clean_text
# VADER needs punctuation, capitals, and exclamation marks

print("Running VADER sentiment scoring...")
print("Scoring 75,593 reviews... this takes 2-3 minutes\n")

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    scores = analyzer.polarity_scores(str(text))
    compound = scores['compound']

    if compound >= 0.05:
        label = 'Positive'
    elif compound <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return compound, label

# Apply to every review
df[['compound_score', 'sentiment_label']] = df['text'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

# Create rating-based sentiment for comparison
def rating_to_sentiment(r):
    if r >= 4.0:
        return 'Positive'
    elif r <= 2.0:
        return 'Negative'
    else:
        return 'Neutral'

df['rating_sentiment'] = df['rating'].apply(rating_to_sentiment)

# Agreement analysis
df['vader_agrees'] = df['sentiment_label'] == df['rating_sentiment']

print("✅ VADER scoring complete")
print("\n── Sentiment Distribution ──────────────────")
sentiment_counts = df['sentiment_label'].value_counts()
total = len(df)
for label, count in sentiment_counts.items():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f"  {label:8} : {bar} {count:,} ({pct:.1f}%)")

print("\n── VADER vs Star Rating Agreement ──────────")
agree_rate = df['vader_agrees'].mean() * 100
print(f"  Agreement rate : {agree_rate:.1f}%")
print(f"  Disagreement   : {100 - agree_rate:.1f}%")

print("\n── Disagreement Breakdown ───────────────────")
disagree = df[df['vader_agrees'] == False]
print(f"  Total disagreements: {len(disagree):,}")
print(f"\n  Where they disagree:")
cross = pd.crosstab(
    disagree['rating_sentiment'],
    disagree['sentiment_label'],
    margins=False
)
print(cross)

print("\n── Sample Reviews ───────────────────────────")
print("\nMost POSITIVE by VADER:")
top_pos = df.nlargest(1, 'compound_score')[['text', 'rating', 'compound_score']]
print(f"  Rating: {top_pos['rating'].values[0]} stars")
print(f"  Score : {top_pos['compound_score'].values[0]}")
print(f"  Text  : {top_pos['text'].values[0][:150]}")

print("\nMost NEGATIVE by VADER:")
top_neg = df.nsmallest(1, 'compound_score')[['text', 'rating', 'compound_score']]
print(f"  Rating: {top_neg['rating'].values[0]} stars")
print(f"  Score : {top_neg['compound_score'].values[0]}")
print(f"  Text  : {top_neg['text'].values[0][:150]}")

Running VADER sentiment scoring...
Scoring 75,593 reviews... this takes 2-3 minutes

✅ VADER scoring complete

── Sentiment Distribution ──────────────────
  Positive : ███████████████████████████████ 47,219 (62.5%)
  Negative : ████████████ 18,902 (25.0%)
  Neutral  : ██████ 9,472 (12.5%)

── VADER vs Star Rating Agreement ──────────
  Agreement rate : 55.3%
  Disagreement   : 44.7%

── Disagreement Breakdown ───────────────────
  Total disagreements: 33,816

  Where they disagree:
sentiment_label   Negative  Neutral  Positive
rating_sentiment                             
Negative                 0     5556     11470
Neutral               3780        0      9031
Positive              2046     1933         0

── Sample Reviews ───────────────────────────

Most POSITIVE by VADER:
  Rating: 5.0 stars
  Score : 0.9999
  Text  : I tried other evolutions and this one is the best!!!!!!!!😂😂😂😂😂😂😂😁😂😂😂😂😂😂😁😂😂😂😂💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💖💟💟💟💟💟💟👏👏👏👏👏👏👏💙💙💙💙💙💚💚💚💚💙💙💗💗💗💗💗💗💖💖💖💗💗💝💞

Most NEGATI

In [12]:
#Save progress and verify dataset

# Save updated dataframe with sentiment columns to Drive
SAVE_PATH = '/content/drive/MyDrive/indata-project/software_reviews_final.csv'
df.to_csv(SAVE_PATH, index=False)
print(f"✅ Progress saved to Google Drive")
print(f"   File: {SAVE_PATH}")

# Quick verification
print(f"\n── Dataset Status ───────────────────────────")
print(f"  Total rows     : {len(df):,}")
print(f"  Total columns  : {df.shape[1]}")
print(f"  Columns        : {df.columns.tolist()}")
print(f"\n── Columns added so far ─────────────────────")
print(f"  clean_text     : cleaned text for LDA")
print(f"  compound_score : VADER score -1.0 to +1.0")
print(f"  sentiment_label: Positive / Negative / Neutral")
print(f"  rating_sentiment: star rating converted to label")
print(f"  vader_agrees   : True if VADER matches star rating")
print(f"\n── Ready for Step 4: LDA Topic Modelling ────")

✅ Progress saved to Google Drive
   File: /content/drive/MyDrive/indata-project/software_reviews_final.csv

── Dataset Status ───────────────────────────
  Total rows     : 75,593
  Total columns  : 21
  Columns        : ['rating', 'review_title', 'text', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase', 'date', 'year', 'month', 'year_month', 'product_name', 'store', 'price', 'average_rating', 'rating_number', 'clean_text', 'compound_score', 'sentiment_label', 'rating_sentiment', 'vader_agrees']

── Columns added so far ─────────────────────
  clean_text     : cleaned text for LDA
  compound_score : VADER score -1.0 to +1.0
  sentiment_label: Positive / Negative / Neutral
  rating_sentiment: star rating converted to label
  vader_agrees   : True if VADER matches star rating

── Ready for Step 4: LDA Topic Modelling ────
